<a href="https://colab.research.google.com/github/edgarbc/My_medium_posts/blob/main/opentelemetry_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logging and tracing using OpenTelemetry

Simple example about telemetry in LLM based applications.

Edgar Bermudez

October, 2025.



In [1]:
!pip install opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp

INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.


In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.resources import Resource

# Configure global tracer provider
resource = Resource.create({"service.name": "medical-ai-demo"})
provider = TracerProvider(resource=resource)
processor = SimpleSpanProcessor(ConsoleSpanExporter())
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

In [5]:
import time, random
from opentelemetry import trace

# Get a tracer instance
tracer = trace.get_tracer(__name__)

def preprocess(data):
    with tracer.start_as_current_span("preprocess") as span:
        time.sleep(random.uniform(0.01, 0.05))
        result = [d * 2 for d in data]
        span.set_attribute("data.len", len(data))
        return result

def inference(data):
    with tracer.start_as_current_span("inference") as span:
        time.sleep(random.uniform(0.02, 0.07))
        predictions = [1 if d > 5 else 0 for d in data]
        span.set_attribute("predictions.sum", sum(predictions))
        return predictions

def postprocess(predictions):
    with tracer.start_as_current_span("postprocess") as span:
        time.sleep(random.uniform(0.01, 0.03))
        report = {"positive_cases": sum(predictions)}
        span.set_attribute("report.size", len(report))
        return report

def run_pipeline(data):
    with tracer.start_as_current_span("pipeline") as span:
        span.set_attribute("input.size", len(data))
        processed = preprocess(data)
        preds = inference(processed)
        report = postprocess(preds)
        span.set_attribute("report", str(report)) # Convert dictionary to string
        return report

result = run_pipeline([1, 2, 3, 6, 8])
print(result)

{
    "name": "preprocess",
    "context": {
        "trace_id": "0x7d1b44c4550963c63c37b518b948c5e6",
        "span_id": "0x97b110f13aa98038",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc80ca41bb3b2e02e",
    "start_time": "2025-10-18T18:24:24.156879Z",
    "end_time": "2025-10-18T18:24:24.187645Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "data.len": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.37.0",
            "service.name": "medical-ai-demo"
        },
        "schema_url": ""
    }
}
{
    "name": "inference",
    "context": {
        "trace_id": "0x7d1b44c4550963c63c37b518b948c5e6",
        "span_id": "0x77c23fc9c0f71e9d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc80ca4

# Trace visualization
If you want to visualize the traces, instead of printing to the console, you need to export the telemetry data and have a backend to visualize the information.

## Install necessary libraries

### Subtask:
Add the OpenTelemetry OTLP exporter library.


**Reasoning**:
Install the `opentelemetry-exporter-otlp` library using pip.



In [7]:
!pip install opentelemetry-exporter-otlp

## Configure otlp exporter

### Subtask:
Replace the `ConsoleSpanExporter` with an `OTLPSpanExporter` to send data to a collector.


**Reasoning**:
Replace the ConsoleSpanExporter with an OTLPSpanExporter to send data to a collector.



In [8]:
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter

# Configure global tracer provider
resource = Resource.create({"service.name": "medical-ai-demo"})
provider = TracerProvider(resource=resource)
# Replace ConsoleSpanExporter with OTLPSpanExporter
otlp_exporter = OTLPSpanExporter()
processor = SimpleSpanProcessor(otlp_exporter)
provider.add_span_processor(processor)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer(__name__)

result = run_pipeline([1, 2, 3, 6, 8])
print(result)

{
    "name": "preprocess",
    "context": {
        "trace_id": "0xd271f4f8987ef1043d62d2ac38bbaad8",
        "span_id": "0x1e5a90fc0b3b4141",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x76780b1612bc95f1",
    "start_time": "2025-10-18T18:26:36.344414Z",
    "end_time": "2025-10-18T18:26:36.355618Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "data.len": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.37.0",
            "service.name": "medical-ai-demo"
        },
        "schema_url": ""
    }
}
{
    "name": "inference",
    "context": {
        "trace_id": "0xd271f4f8987ef1043d62d2ac38bbaad8",
        "span_id": "0x8204134984631fac",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x76780b

## (optional) run a tracing backend

### Subtask:
Provide instructions or code to set up a local Jaeger or Zipkin instance for visualization.


**Reasoning**:
Provide instructions to set up a local Jaeger instance using Docker.



In [10]:
# Instructions to set up a local Jaeger instance using Docker:
# 1. Ensure you have Docker installed on your system.
# 2. Open your terminal or command prompt.
# 3. Pull and run the Jaeger all-in-one image, mapping the OTLP gRPC port (4317) and the UI port (16686) to your local machine.
#    The following command will pull the image if not already present and run it:
# docker run -d --name jaeger -p 16686:16686 -p 4317:4317 jaegertracing/all-in-one:latest
# 4. Once the container is running, you can access the Jaeger UI by opening a web browser and navigating to http://localhost:16686.

**Reasoning**:
Execute the code cell that calls the `run_pipeline` function to generate and send telemetry data using the configured OTLP exporter.



In [11]:
result = run_pipeline([1, 2, 3, 6, 8])
print(result)

{
    "name": "preprocess",
    "context": {
        "trace_id": "0x1a79c56b337d0cb86387879938fda92d",
        "span_id": "0xa1f5d3d4c3ed11cf",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6104ea7e66e64bfa",
    "start_time": "2025-10-18T18:27:46.271973Z",
    "end_time": "2025-10-18T18:27:46.300451Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "data.len": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.37.0",
            "service.name": "medical-ai-demo"
        },
        "schema_url": ""
    }
}
{
    "name": "inference",
    "context": {
        "trace_id": "0x1a79c56b337d0cb86387879938fda92d",
        "span_id": "0xae78db81b731a857",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6104ea